# Week 5 — LSTM Model
FaultLine: trains an LSTM on 60-day sequences of the Week 2 features to predict
the same target as ETAS and XGBoost -- whether a fault section will have a
qualifying (M4.0+) event within the next 30 days.

**What's different from XGBoost, and why:**
1. **Sequences, not rows.** Each sample is the previous `SEQ_LEN` days of
   features for one segment, so the model can learn from how features *trend*.
2. **NaNs must be filled.** XGBoost routes missing values natively; an LSTM
   can't. We fill with the training median and add a `<col>_missing` 0/1 flag
   so the model knows the filled value isn't real.
3. **Features must be scaled.** Neural nets are sensitive to input scale; every
   numeric column is z-scored.
4. **Fill values and scaling are fit on training rows only** -- same leakage
   logic as computing `scale_pos_weight` from training data only in XGBoost.

Target, split dates, embargo, metrics, and output format are intentionally
identical to the XGBoost notebook so the comparison stays fair.

**Setup (run once in a terminal, with `.venv` activated):**
```
python -m pip install torch
```


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.metrics import (
    precision_score, recall_score, average_precision_score,
    brier_score_loss, roc_auc_score,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
pd.set_option("display.max_columns", None)


## Configuration

`SEQ_LEN = 60`: the features already contain 7/30/90-day rolling summaries, so
the LSTM doesn't need a huge window to see long-range context -- 60 days lets it
see how those summaries are trending. This is a good ablation knob later (30
vs. 60 vs. 90).

In [ ]:
MAG_THRESHOLD = 4.0
FORECAST_DAYS = 30

TRAIN_END = pd.Timestamp("2021-12-31")
VALIDATE_START = pd.Timestamp("2022-01-01")
VALIDATE_END = pd.Timestamp("2022-12-31")
TEST_START = pd.Timestamp("2023-01-01")
TEST_END = pd.Timestamp("2025-12-31")

PROJECTED_CRS = "EPSG:3310"
MAX_ASSIGN_DIST_KM = 20

SEQ_LEN = 60
HIDDEN_SIZE = 64
NUM_LAYERS = 1
DROPOUT = 0.2
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
MAX_EPOCHS = 40
PATIENCE = 5          # early stopping: stop after this many epochs without validation improvement

NUMERIC_COLS = [
    "seismicity_rate_7d", "seismicity_rate_30d", "seismicity_rate_90d", "seismicity_rate_30d_z",
    "b_value", "b_value_se", "n_events", "gps_strain_rate", "gps_station_distance_km",
    "max_mag_7d", "max_mag_30d", "days_since_last_m35",
]
BOOL_COLS = ["b_value_reliable", "gps_reliable"]


## Load Week 2 features -- with guards against the bugs we've already hit

Two checks run immediately on load:
- **Timezone:** strip it if present (the same UTC-vs-naive bug hit twice before).
- **Stale-file guard:** `gps_strain_rate` must have nonzero variance. A constant
  column is exactly what the stale `features.parquet` produced last time, and
  it passes every NaN check silently. If this assert fails, re-run
  `02_feature_engineering.ipynb` with Restart & Run All.

In [3]:
features = pd.read_parquet("../data/processed/features.parquet")
features["date"] = pd.to_datetime(features["date"])
if features["date"].dt.tz is not None:
    features["date"] = features["date"].dt.tz_localize(None)

print(features["gps_strain_rate"].describe())
assert features["gps_strain_rate"].std() > 0, (
    "gps_strain_rate has zero variance -- features.parquet is stale. "
    "Restart & Run All in 02_feature_engineering.ipynb, then re-run this notebook."
)
print(features.shape)


count    36520.000000
mean        27.670689
std         14.642717
min          0.000000
25%         19.380615
50%         30.711634
75%         38.244993
max         66.761948
Name: gps_strain_rate, dtype: float64
(36520, 14)


## Rebuild the event-level catalog and the forward-looking label

Identical to the XGBoost notebook. `features.parquet` only has aggregated
counts, so the M4.0+ label needs individual event magnitudes.

In [4]:
faults = gpd.read_file("../data/raw/san_andreas_faults.geojson")
faults = faults.dissolve(by="section_na").reset_index()
faults["fault_segment_id"] = faults.index.astype(str)

quakes = pd.read_parquet("../data/raw/usgs_2016_2026.parquet")
quakes["time"] = pd.to_datetime(quakes["time"])
if quakes["time"].dt.tz is not None:
    quakes["time"] = quakes["time"].dt.tz_localize(None)
# NOTE: check quakes.columns.tolist() -- this project's schema has varied between
# 'mag'/'magnitud' and 'magnitude'/'event_id'. Adjust the rename if needed.
quakes = quakes.rename(columns={"event_id": "id", "mag": "magnitude"})

quake_points_m = gpd.GeoDataFrame(
    quakes.copy(), geometry=gpd.points_from_xy(quakes["longitude"], quakes["latitude"]), crs="EPSG:4326"
).to_crs(PROJECTED_CRS)
faults_m = faults.to_crs(PROJECTED_CRS)
joined = gpd.sjoin_nearest(
    quake_points_m, faults_m[["fault_segment_id", "geometry"]],
    max_distance=MAX_ASSIGN_DIST_KM * 1000, distance_col="dist_to_fault_m",
)
quakes_assigned = joined.drop(columns="geometry").rename(columns={"time": "event_time"})


def build_labels(features_dates, quakes_assigned, mag_threshold, forecast_days):
    qualifying = quakes_assigned[quakes_assigned["magnitude"] >= mag_threshold].copy()
    qualifying["event_date"] = qualifying["event_time"].dt.floor("D")
    labels = []
    for seg_id, seg_dates in features_dates.groupby("fault_segment_id"):
        seg_events = qualifying[qualifying["fault_segment_id"] == seg_id]["event_date"].sort_values().values
        for d in seg_dates["date"]:
            window_end = d + pd.Timedelta(days=forecast_days)
            has_event = np.any((seg_events > np.datetime64(d)) & (seg_events <= np.datetime64(window_end)))
            labels.append({"fault_segment_id": seg_id, "date": d, "label": int(has_event)})
    return pd.DataFrame(labels)


labels = build_labels(features[["fault_segment_id", "date"]], quakes_assigned, MAG_THRESHOLD, FORECAST_DAYS)
data = features.merge(labels, on=["fault_segment_id", "date"], how="left")
print(f"Overall positive rate: {data['label'].mean():.4f}")


Overall positive rate: 0.0179


## Assign splits -- but DON'T drop embargo rows yet

The sequence builder needs each segment's dates to be contiguous. If we deleted
the 30-day embargo gaps first, a window could silently stitch together days on
either side of a hole as if they were consecutive. So: label every row's split
now, build windows on the full unbroken timeline, and drop the *windows* whose
forecast date is in an embargo period afterwards.

(A validation window may include feature values from the end of the training
period -- that's fine. Features only look backward; the leak we guard against is
in the *label*, which the embargo already handles.)

In [5]:
def assign_split(date, forecast_days, train_end, validate_start, validate_end, test_start, test_end):
    if date <= train_end - pd.Timedelta(days=forecast_days):
        return "train"
    elif validate_start <= date <= validate_end - pd.Timedelta(days=forecast_days):
        return "validate"
    elif test_start <= date <= test_end - pd.Timedelta(days=forecast_days):
        return "test"
    else:
        return "embargo"

data["split"] = data["date"].apply(
    lambda d: assign_split(d, FORECAST_DAYS, TRAIN_END, VALIDATE_START, VALIDATE_END, TEST_START, TEST_END)
)
print(data["split"].value_counts())


split
train       21610
test        10660
validate     3350
embargo       900
Name: count, dtype: int64


## Preprocess: fill, flag, scale (fit on training rows only), one-hot segments

`fault_segment_id` is one-hot encoded and appended to every timestep. XGBoost
took it as a native categorical; an LSTM needs numbers, and with only 10
segments, one-hot is simple and transparent.

In [6]:
def fit_preprocessor(train_df, numeric_cols):
    """Learn fill values and scaling from TRAINING rows only."""
    params = {}
    for col in numeric_cols:
        s = train_df[col].astype(float)
        median = float(s.median()) if s.notna().any() else 0.0
        filled = s.fillna(median)
        std = float(filled.std())
        params[col] = {
            "median": median,
            "mean": float(filled.mean()),
            "std": std if std > 0 else 1.0,
            "add_missing_flag": bool(s.isna().any()),
        }
    return params


def apply_preprocessor(df, params):
    out = df.copy()
    for col, p in params.items():
        if p["add_missing_flag"]:
            out[f"{col}_missing"] = out[col].isna().astype(np.float32)
        out[col] = (out[col].astype(float).fillna(p["median"]) - p["mean"]) / p["std"]
    return out


prep_params = fit_preprocessor(data[data["split"] == "train"], NUMERIC_COLS)
data_prep = apply_preprocessor(data, prep_params)
for col in BOOL_COLS:
    data_prep[col] = data_prep[col].astype(np.float32)

segment_ids = sorted(data_prep["fault_segment_id"].astype(str).unique(), key=int)
seg_onehot = pd.get_dummies(data_prep["fault_segment_id"].astype(str), prefix="seg").reindex(
    columns=[f"seg_{s}" for s in segment_ids], fill_value=0
).astype(np.float32)
data_prep = pd.concat([data_prep, seg_onehot], axis=1)

missing_flag_cols = [f"{c}_missing" for c, p in prep_params.items() if p["add_missing_flag"]]
INPUT_COLS = NUMERIC_COLS + missing_flag_cols + BOOL_COLS + list(seg_onehot.columns)
print(f"{len(INPUT_COLS)} input features per timestep:")
print(INPUT_COLS)
assert not data_prep[INPUT_COLS].isna().any().any(), "NaN survived preprocessing"


23 input features per timestep:
['seismicity_rate_7d', 'seismicity_rate_30d', 'seismicity_rate_90d', 'seismicity_rate_30d_z', 'b_value', 'b_value_se', 'n_events', 'gps_strain_rate', 'gps_station_distance_km', 'b_value_missing', 'b_value_se_missing', 'b_value_reliable', 'gps_reliable', 'seg_0', 'seg_1', 'seg_2', 'seg_3', 'seg_4', 'seg_5', 'seg_6', 'seg_7', 'seg_8', 'seg_9']


## Build 60-day sequences

Tested against synthetic data before use: every window ends exactly **on** its
own forecast date (no peeking forward), windows never mix two segments, and the
first `SEQ_LEN - 1` days of each segment are dropped because they don't have a
full window of history yet.

In [7]:
def build_sequences(df, feature_cols, seq_len):
    X_parts, meta_parts = [], []
    for seg_id, seg in df.groupby("fault_segment_id", observed=True):
        seg = seg.sort_values("date")
        gaps = seg["date"].diff().dropna().dt.days
        assert (gaps == 1).all(), f"segment {seg_id} has non-contiguous dates"
        F = seg[feature_cols].to_numpy(dtype=np.float32)
        if len(F) < seq_len:
            continue
        windows = np.lib.stride_tricks.sliding_window_view(F, seq_len, axis=0)  # (n, n_feat, L)
        X_parts.append(windows.transpose(0, 2, 1))                              # (n, L, n_feat)
        meta_parts.append(seg.iloc[seq_len - 1:][["fault_segment_id", "date", "label", "split"]])
    return np.ascontiguousarray(np.concatenate(X_parts)), pd.concat(meta_parts, ignore_index=True)


X_all, meta_all = build_sequences(data_prep, INPUT_COLS, SEQ_LEN)

def take(split):
    mask = (meta_all["split"] == split).to_numpy()
    return X_all[mask], meta_all.loc[mask, "label"].to_numpy(dtype=np.float32), meta_all[mask].reset_index(drop=True)

X_train, y_train, meta_train = take("train")
X_validate, y_validate, meta_validate = take("validate")
X_test, y_test, meta_test = take("test")

for name, X, y in [("Train", X_train, y_train), ("Validate", X_validate, y_validate), ("Test", X_test, y_test)]:
    print(f"{name}: {X.shape}, positive rate {y.mean():.4f}")


Train: (21020, 60, 23), positive rate 0.0186
Validate: (3350, 60, 23), positive rate 0.0179
Test: (10660, 60, 23), positive rate 0.0188


## Define and train the model

- The LSTM reads the whole 60-day window; its final hidden state summarizes it,
  and a linear layer turns that into one logit (probability after sigmoid).
- `BCEWithLogitsLoss(pos_weight=...)` is the PyTorch equivalent of XGBoost's
  `scale_pos_weight` -- computed from training labels only.
- **Early stopping on validation PR-AUC:** after each epoch we score the
  validation set; the best epoch's weights are kept, and training stops after
  `PATIENCE` epochs with no improvement. This keeps us from overfitting to the
  training years.

In [8]:
class FaultLineLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, num_layers=num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)                              # h_n: (num_layers, batch, hidden)
        return self.head(self.dropout(h_n[-1])).squeeze(-1)     # raw logits


def predict_proba(model, X, batch_size=1024):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            out.append(torch.sigmoid(model(torch.from_numpy(X[i:i + batch_size]))).numpy())
    return np.concatenate(out)


model = FaultLineLSTM(n_features=len(INPUT_COLS))
pos_weight = torch.tensor((y_train == 0).sum() / max((y_train == 1).sum(), 1), dtype=torch.float32)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
    batch_size=BATCH_SIZE, shuffle=True,
)
print(f"pos_weight: {pos_weight.item():.2f}")

best_ap, best_state, epochs_without_improvement = -1.0, None, 0
history = []
for epoch in range(MAX_EPOCHS):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    val_ap = average_precision_score(y_validate, predict_proba(model, X_validate))
    history.append({"epoch": epoch, "train_loss": total_loss / len(X_train), "val_pr_auc": val_ap})
    print(f"epoch {epoch:2d}  train_loss {total_loss / len(X_train):.4f}  val_PR-AUC {val_ap:.3f}")

    if val_ap > best_ap:
        best_ap, epochs_without_improvement = val_ap, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val PR-AUC {best_ap:.3f})")
            break

model.load_state_dict(best_state)


pos_weight: 52.76
epoch  0  train_loss 1.0565  val_PR-AUC 0.344
epoch  1  train_loss 0.7033  val_PR-AUC 0.119
epoch  2  train_loss 0.6026  val_PR-AUC 0.146
epoch  3  train_loss 0.4759  val_PR-AUC 0.288
epoch  4  train_loss 0.5031  val_PR-AUC 0.126
epoch  5  train_loss 0.4307  val_PR-AUC 0.101
Early stopping at epoch 5 (best val PR-AUC 0.344)


<All keys matched successfully>

## Evaluate -- same metrics as XGBoost

In [9]:
def evaluate(y, probs, split_name):
    preds = (probs >= 0.5).astype(int)
    print(f"--- {split_name} ---")
    print(f"Positive rate: {y.mean():.4f}")
    print(f"Precision: {precision_score(y, preds, zero_division=0):.3f}")
    print(f"Recall: {recall_score(y, preds, zero_division=0):.3f}")
    print(f"PR-AUC: {average_precision_score(y, probs):.3f}")
    print(f"ROC-AUC: {roc_auc_score(y, probs):.3f}")
    print(f"Brier score (lower is better-calibrated): {brier_score_loss(y, probs):.4f}")
    print()

validate_probs = predict_proba(model, X_validate)
test_probs = predict_proba(model, X_test)
evaluate(y_validate, validate_probs, "Validate")
evaluate(y_test, test_probs, "Test")


--- Validate ---
Positive rate: 0.0179
Precision: 0.040
Recall: 1.000
PR-AUC: 0.344
ROC-AUC: 0.983
Brier score (lower is better-calibrated): 0.2444

--- Test ---
Positive rate: 0.0188
Precision: 0.022
Recall: 0.710
PR-AUC: 0.028
ROC-AUC: 0.600
Brier score (lower is better-calibrated): 0.3308



In [12]:
check = meta_test[["fault_segment_id", "label"]].copy()
check["fault_segment_id"] = check["fault_segment_id"].astype(str)
raw = test_probs
odds = (raw / (1 - raw)) / pos_weight.item()      # undo the pos_weight inflation
check["deweighted_prob"] = odds / (1 + odds)

summary = check.groupby("fault_segment_id").agg(
    actual_rate=("label", "mean"),
    mean_predicted=("deweighted_prob", "mean"),
).merge(faults[["fault_segment_id", "section_na"]], on="fault_segment_id")
print(summary.sort_values("actual_rate", ascending=False))

  fault_segment_id  actual_rate  mean_predicted  \
7                7     0.075047        0.019141   
8                8     0.056285        0.052429   
2                2     0.028143        0.089264   
5                5     0.028143        0.023033   
1                1     0.000000        0.055575   
0                0     0.000000        0.021464   
4                4     0.000000        0.021143   
3                3     0.000000        0.016515   
6                6     0.000000        0.019935   
9                9     0.000000        0.020081   

                         section_na  
7  San Bernardino Mountains section  
8      Santa Cruz Mountains section  
2                  Creeping section  
5                 Parkfield section  
1                 Coachella section  
0           Cholame-Carrizo section  
4               North Coast section  
3                    Mojave section  
6                 Peninsula section  
9              Shelter Cove Section  


In [13]:
qa = quakes_assigned[quakes_assigned["magnitude"] >= MAG_THRESHOLD].copy()
qa["period"] = np.where(qa["event_time"] <= TRAIN_END, "train (2016-2021)",
               np.where(qa["event_time"] < TEST_START, "validate (2022)", "test (2023-2025)"))
event_counts = (qa.groupby(["fault_segment_id", "period"]).size()
                  .unstack(fill_value=0)
                  .merge(faults[["fault_segment_id", "section_na"]], left_index=True, right_on="fault_segment_id"))
print(event_counts)

   test (2023-2025)  train (2016-2021)  validate (2022) fault_segment_id  \
1                 0                  8                0                1   
2                 1                  6                2                2   
5                 1                  1                0                5   
7                 3                  2                0                7   
8                 2                  2                0                8   

                         section_na  
1                 Coachella section  
2                  Creeping section  
5                 Parkfield section  
7  San Bernardino Mountains section  
8      Santa Cruz Mountains section  


## Permutation importance -- a first look at whether strain rate matters

LSTMs don't have built-in feature importance like XGBoost. Permutation
importance is the standard substitute: shuffle one feature across samples
(breaking its link to the label), re-score, and measure how much PR-AUC drops.
A big drop means the model relies on that feature. The dedicated with/without
strain-rate ablation (retraining without the feature) is still the real test,
but this is a cheap first look.

In [10]:
rng = np.random.default_rng(SEED)
baseline_ap = average_precision_score(y_validate, validate_probs)
importance = {}
for j, col in enumerate(INPUT_COLS):
    if col.startswith("seg_"):
        continue
    X_perm = X_validate.copy()
    X_perm[:, :, j] = X_perm[rng.permutation(len(X_perm)), :, j]
    importance[col] = baseline_ap - average_precision_score(y_validate, predict_proba(model, X_perm))

print(f"Baseline validation PR-AUC: {baseline_ap:.3f}")
print(pd.Series(importance).sort_values(ascending=False))


Baseline validation PR-AUC: 0.344
n_events                   0.108049
b_value                    0.090352
seismicity_rate_30d        0.069586
gps_strain_rate            0.064831
seismicity_rate_30d_z      0.049793
b_value_reliable           0.041637
gps_station_distance_km    0.030038
b_value_se_missing        -0.000181
b_value_missing           -0.009265
seismicity_rate_90d       -0.015793
gps_reliable              -0.038696
seismicity_rate_7d        -0.086996
b_value_se                -0.185459
dtype: float64


## Save the forecast, the model, and the preprocessing parameters

The preprocessing parameters (fill medians, means, stds, column order) are saved
alongside the model because any future inference -- the ensemble, or a FastAPI
endpoint -- must apply *exactly* the same transformation to new data, or the
model's inputs won't mean what they meant during training.

In [11]:
latest_date = meta_test["date"].max()
latest_mask = (meta_test["date"] == latest_date).to_numpy()
lstm_forecast = meta_test.loc[latest_mask, ["fault_segment_id"]].copy()
lstm_forecast["probability_30d"] = test_probs[latest_mask]
lstm_forecast["fault_segment_id"] = lstm_forecast["fault_segment_id"].astype(str)
lstm_forecast = lstm_forecast.merge(
    faults[["fault_segment_id", "section_na"]], on="fault_segment_id"
).sort_values("probability_30d", ascending=False)

print(f"Forecast as of {latest_date.date()}:")
print(lstm_forecast)

os.makedirs("../models", exist_ok=True)
lstm_forecast.to_parquet("../data/processed/lstm_forecast.parquet", index=False)
torch.save(model.state_dict(), "../models/lstm_model.pt")
with open("../models/lstm_preprocessing.json", "w") as f:
    json.dump({"seq_len": SEQ_LEN, "input_cols": INPUT_COLS, "numeric_params": prep_params,
               "segment_ids": segment_ids}, f, indent=2)
print("\nSaved forecast, model weights, and preprocessing parameters.")


Forecast as of 2025-12-01:
  fault_segment_id  probability_30d                        section_na
2                2         0.815328                  Creeping section
1                1         0.760690                 Coachella section
3                3         0.748405                    Mojave section
0                0         0.748217           Cholame-Carrizo section
5                5         0.716411                 Parkfield section
7                7         0.685251  San Bernardino Mountains section
4                4         0.679628               North Coast section
8                8         0.675263      Santa Cruz Mountains section
9                9         0.198217              Shelter Cove Section
6                6         0.109647                 Peninsula section

Saved forecast, model weights, and preprocessing parameters.
